# Test Case 1 - Single Capacitor

Übertragungsfunktion von der Eingangsspannung `Uq` zur Kondensatorspannung `U_C`.


In [ ]:
from CircuitCalculator.Circuit.transfer_function import symbolic_transfer_function, numeric_transfer_function, TransferFunctionOutput
from CircuitCalculator.Circuit.circuit import Circuit
from CircuitCalculator.Circuit.Components import components as cmp
from scipy import signal
from IPython.display import display
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt


## Setup Circuit


In [ ]:
R1, R2, R3 = 10, 20, 30
C = 1e-3
V0 = 5
circuit = Circuit(
    components=[
        cmp.dc_voltage_source(id='Uq', V=V0, nodes=('1', '0')),
        cmp.resistor(id='R1', R=R1, nodes=('1', '2')),
        cmp.resistor(id='R2', R=R2, nodes=('2', '0')),
        cmp.resistor(id='R3', R=R3, nodes=('2', '3')),
        cmp.capacitor(id='C', C=C, nodes=('3', '0'))
    ],
    ground_node='0'
)
circuit


## Configure Transfer Functions


In [ ]:
analyses = [
    {
        'title': 'H_Uq->Uc(s)',
        'input_id': 'Uq',
        'output': TransferFunctionOutput.voltage('C'),
        'output_label': 'U_C',
    },
]
analyses


## Calculate Symbolic And Numeric Transfer Functions


In [ ]:
results = []
for analysis in analyses:
    tf_symbolic = symbolic_transfer_function(circuit, input_id=analysis["input_id"], output=analysis["output"])
    tf_numeric = numeric_transfer_function(circuit, input_id=analysis["input_id"], output=analysis["output"])
    results.append({**analysis, "symbolic": tf_symbolic, "numeric": tf_numeric})
results


## Show Formulas


In [ ]:
for result in results:
    print(result["title"])
    display(sp.Eq(sp.Symbol(result["title"]), result["symbolic"].expr))
    print("Numerator:", result["symbolic"].numerator())
    print("Denominator:", result["symbolic"].denominator())
    print("Zeros:", result["symbolic"].zeros() if result["symbolic"].zeros() else "keine")
    print("Poles:", result["symbolic"].poles())
    print("Numeric numerator coefficients:", result["numeric"].numerator_coeffs)
    print("Numeric denominator coefficients:", result["numeric"].denominator_coeffs)
    print()


## Bode Diagram


In [ ]:
w = np.logspace(-1, 4, 500)
fig, ax = plt.subplots(nrows=2, ncols=len(results), figsize=(7 * len(results), 6), sharex="col")
if len(results) == 1:
    ax = np.array(ax).reshape(2, 1)
for idx, result in enumerate(results):
    system = result["numeric"].transfer_function()
    w_bode, magnitude_db, phase_deg = signal.bode(system, w=w)
    ax[0, idx].semilogx(w_bode, magnitude_db, color="tab:blue")
    ax[0, idx].set_title(result["title"])
    ax[0, idx].set_ylabel("Magnitude / dB")
    ax[0, idx].grid(True, which="both")
    ax[1, idx].semilogx(w_bode, phase_deg, color="tab:orange")
    ax[1, idx].set_xlabel("ω / rad/s")
    ax[1, idx].set_ylabel("Phase / deg")
    ax[1, idx].grid(True, which="both")
plt.tight_layout()
plt.show()


## Ortskurve


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    ax = np.array([ax])
for idx, result in enumerate(results):
    system = result["numeric"].transfer_function()
    w_resp, response = signal.freqresp(system, w=w)
    ax[idx].plot(response.real, response.imag, color="tab:green")
    ax[idx].scatter([response.real[0], response.real[-1]], [response.imag[0], response.imag[-1]], color=["tab:blue", "tab:red"], zorder=3)
    ax[idx].axhline(0, color="0.5", linewidth=0.8)
    ax[idx].axvline(0, color="0.5", linewidth=0.8)
    ax[idx].set_xlabel("Re{H(jω)}")
    ax[idx].set_ylabel("Im{H(jω)}")
    ax[idx].set_title(result["title"])
    ax[idx].grid(True)
    ax[idx].set_aspect("equal", "box")
plt.tight_layout()
plt.show()


## Pole-Zero Diagram


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    ax = np.array([ax])
for idx, result in enumerate(results):
    zero_values = result["numeric"].zeros()
    pole_values = result["numeric"].poles()
    if zero_values.size > 0:
        ax[idx].scatter(zero_values.real, zero_values.imag, marker="o", s=100, facecolors="none", edgecolors="tab:blue", label="Zeros")
    ax[idx].scatter(pole_values.real, pole_values.imag, marker="x", s=100, color="tab:red", label="Poles")
    ax[idx].axhline(0, color="0.5", linewidth=0.8)
    ax[idx].axvline(0, color="0.5", linewidth=0.8)
    ax[idx].set_xlabel("Re{s}")
    ax[idx].set_ylabel("Im{s}")
    ax[idx].set_title(result["title"])
    ax[idx].grid(True)
    ax[idx].legend()
plt.tight_layout()
plt.show()
